Copyright (c) Microsoft Corporation. All rights reserved.

Licensed under the MIT License.

![Impressions](https://PixelServer20190423114238.azurewebsites.net/api/impressions/MachineLearningNotebooks/how-to-use-azureml/automated-machine-learning/dataprep-remote-execution/auto-ml-dataprep-remote-execution.png)

# Automated Machine Learning
_**Prepare Data using `azureml.dataprep` for Remote Execution (AmlCompute)**_

## Contents
1. [Introduction](#Introduction)
1. [Setup](#Setup)
1. [Data](#Data)
1. [Train](#Train)
1. [Results](#Results)
1. [Test](#Test)

## Introduction
In this example we showcase how you can use the `azureml.dataprep` SDK to load and prepare data for AutoML. `azureml.dataprep` can also be used standalone; full documentation can be found [here](https://github.com/Microsoft/PendletonDocs).

Make sure you have executed the [configuration](../../../configuration.ipynb) before running this notebook.

In this notebook you will learn how to:
1. Define data loading and preparation steps in a `Dataflow` using `azureml.dataprep`.
2. Pass the `Dataflow` to AutoML for a local run.
3. Pass the `Dataflow` to AutoML for a remote run.

## Setup

Currently, Data Prep only supports __Ubuntu 16__ and __Red Hat Enterprise Linux 7__. We are working on supporting more linux distros.

As part of the setup you have already created an Azure ML `Workspace` object. For AutoML you will need to create an `Experiment` object, which is a named object in a `Workspace` used to run experiments.

In [ ]:
import logging

import pandas as pd

import azureml.core
from azureml.core.experiment import Experiment
from azureml.core.workspace import Workspace
from azureml.core.dataset import Dataset
from azureml.train.automl import AutoMLConfig


In [ ]:
ws = Workspace.from_config()
 
# choose a name for experiment
experiment_name = 'automl-streaming-sampleweight'
# project folder
project_folder = './sample_projects/automl-streaming-sampleweight'
 
experiment = Experiment(ws, experiment_name)
 
output = {}
output['SDK version'] = azureml.core.VERSION
output['Subscription ID'] = ws.subscription_id
output['Workspace Name'] = ws.name
output['Resource Group'] = ws.resource_group
output['Location'] = ws.location
output['Project Directory'] = project_folder
output['Experiment Name'] = experiment.name
pd.set_option('display.max_colwidth', -1)
outputDf = pd.DataFrame(data = output, index = [''])
outputDf.T

## Data

In [ ]:
sample_weight = Dataset.Tabular.from_delimited_files("https://daholstestorage6962a53f8.blob.core.windows.net/data/dataprep_weight.csv")
sample_weight_valid = Dataset.Tabular.from_delimited_files("https://daholstestorage6962a53f8.blob.core.windows.net/data/dataprep_weight.csv")
print(sample_weight.take(5).to_pandas_dataframe())
print(sample_weight_valid.take(5).to_pandas_dataframe())

### Review the Data Preparation Result

You can peek the result of a Dataflow at any range using `skip(i)` and `head(j)`. Doing so evaluates only `j` records for all the steps in the Dataflow, which makes it fast even against large datasets.

`Dataflow` objects are immutable and are composed of a list of data preparation steps. A `Dataflow` object can be branched at any point for further usage.

## Train

This creates a general AutoML settings object applicable for both local and remote runs.

In [ ]:
automl_settings = {
    "iteration_timeout_minutes" : 10,
    "iterations" : 5,
    "primary_metric" : 'accuracy',
    "preprocess" : False,
    "verbosity" : logging.INFO,
    "force_streaming": True,    
    "use_incremental_learning_override": True,
    "enable_ensembling" : False,
    "enable_stack_ensemble" : False,
    "enable_voting_ensemble" : False
}

### Create or Attach an AmlCompute cluster

In [ ]:
from azureml.core.compute import AmlCompute
from azureml.core.compute import ComputeTarget

# Choose a name for your cluster.
amlcompute_cluster_name = "cpu-cluster"

found = False

# Check if this compute target already exists in the workspace.

cts = ws.compute_targets
if amlcompute_cluster_name in cts and cts[amlcompute_cluster_name].type == 'AmlCompute':
    found = True
    print('Found existing compute target.')
    compute_target = cts[amlcompute_cluster_name]

if not found:
    print('Creating a new compute target...')
    provisioning_config = AmlCompute.provisioning_configuration(vm_size = "STANDARD_D2_V2", # for GPU, use "STANDARD_NC6"
                                                                #vm_priority = 'lowpriority', # optional
                                                                max_nodes = 6)

    # Create the cluster.\n",
    compute_target = ComputeTarget.create(ws, amlcompute_cluster_name, provisioning_config)

    # Can poll for a minimum number of nodes and for a specific timeout.
    # If no min_node_count is provided, it will use the scale settings for the cluster.
    compute_target.wait_for_completion(show_output = True, min_node_count = None, timeout_in_minutes = 20)

     # For a more detailed view of current AmlCompute status, use get_status().

In [ ]:
from azureml.core.runconfig import RunConfiguration
from azureml.core.conda_dependencies import CondaDependencies

# create a new RunConfig object
conda_run_config = RunConfiguration(framework="python")

# Set compute target to AmlCompute
conda_run_config.target = compute_target
conda_run_config.environment.docker.enabled = True
conda_run_config.environment.docker.base_image = azureml.core.runconfig.DEFAULT_CPU_IMAGE

cd = CondaDependencies.create(conda_packages=['numpy','py-xgboost<=0.80'])
#cd.set_pip_option("--extra-index-url https://pypi.org/simple --extra-index-url https://azuremlsdktestpypi.azureedge.net/Automl-UX-Runner-AzuremlCli-Sdk-Gated-CI/4220796/ --extra-index-url https://azuremlsdktestpypi.azureedge.net/AzureML-Train-AutoML-Validation/4479955/")

# we need to pin these versions due to bugfixes. will remove once the changes make to pypi
#cd.add_pip_package("azureml-dataprep==1.1.10a2019073102")
#cd.add_pip_package("nimbusml==1.2.1")# we need to pin these versions due to bugfixes. will remove once the changes make to pypi
#cd.add_pip_package("azureml-sdk[automl]<0.1.1")
#cd.add_pip_package("azureml-defaults<0.1.1")
#cd.add_pip_package("azureml-train-automl<0.1.1")

rc = RunConfiguration(conda_dependencies=cd)
conda_run_config.environment.python.conda_dependencies = cd

In [ ]:
print(conda_run_config)

### Pass Data with `Dataflow` Objects

The `Dataflow` objects captured above can also be passed to the `submit` method for a remote run. AutoML will serialize the `Dataflow` object and send it to the remote compute target. The `Dataflow` will not be evaluated locally.

In [ ]:
automl_config = AutoMLConfig(task = 'classification',
                             debug_log = 'automl_errors.log',
                             path = project_folder,
                             run_configuration=conda_run_config,
                             training_data = sample_weight,
                             validation_data = sample_weight,
                             label_column_name = '1.24',
                             weight_column_name = 'sample_weight',
                             **automl_settings)

In [ ]:
remote_run = experiment.submit(automl_config, show_output = False)

In [ ]:
remote_run

### Pre-process cache cleanup
The preprocess data gets cache at user default file store. When the run is completed the cache can be cleaned by running below cell

In [ ]:
from azureml.train.automl.run import AutoMLRun
remote_run2 = AutoMLRun(experiment, 'AutoML_88ef270f-5176-4f1b-8e43-4e088e27a6ef')

In [ ]:
remote_run.clean_preprocessor_cache()

### Cancelling Runs
You can cancel ongoing remote runs using the `cancel` and `cancel_iteration` functions.

In [ ]:
# Cancel the ongoing experiment and stop scheduling new iterations.
# remote_run.cancel()

# Cancel iteration 1 and move onto iteration 2.
# remote_run.cancel_iteration(1)

## Results

#### Widget for Monitoring Runs

The widget will first report a "loading" status while running the first iteration. After completing the first iteration, an auto-updating graph and table will be shown. The widget will refresh once per minute, so you should see the graph update as child runs complete.

**Note:** The widget displays a link at the bottom. Use this link to open a web interface to explore the individual run details.

In [ ]:
from azureml.widgets import RunDetails
RunDetails(remote_run).show()

#### Retrieve All Child Runs
You can also use SDK methods to fetch all the child runs and see individual metrics that we log.

In [ ]:
children = list(remote_run.get_children())
metricslist = {}
for run in children:
    properties = run.get_properties()
    metrics = {k: v for k, v in run.get_metrics().items() if isinstance(v, float)}
    metricslist[int(properties['iteration'])] = metrics
    
rundata = pd.DataFrame(metricslist).sort_index(1)
rundata

### Retrieve the Best Model

Below we select the best pipeline from our iterations. The `get_output` method returns the best run and the fitted model. Overloads on `get_output` allow you to retrieve the best run and fitted model for *any* logged metric or for a particular *iteration*.

In [ ]:
best_run, fitted_model = remote_run2.get_output()
print(best_run)
print(fitted_model)

#### Best Model Based on Any Other Metric
Show the run and the model that has the smallest `log_loss` value:

In [ ]:
lookup_metric = "log_loss"
best_run, fitted_model = remote_run.get_output(metric = lookup_metric)
print(best_run)
print(fitted_model)

#### Model from a Specific Iteration
Show the run and the model from the first iteration:

In [ ]:
iteration = 0
best_run, fitted_model = remote_run.get_output(iteration = iteration)
print(best_run)
print(fitted_model)

## Test

#### Load Test Data
For the test data, it should have the same preparation step as the train data. Otherwise it might get failed at the preprocessing step.

In [ ]:
dataset_test = Dataset.Tabular.from_delimited_files('https://daholstestorage6962a53f8.blob.core.windows.net/data/1MB_encoded.csv')

#### Testing Our Best Fitted Model
We will use confusion matrix to see how our model works.

In [ ]:
from pandas_ml import ConfusionMatrix

y_test = dataset_test.keep_columns(columns=['1.24']).to_pandas_dataframe()
X_test = dataset_test.drop_columns(columns=['1.24']).to_pandas_dataframe()


ypred = fitted_model.predict(X_test)

cm = ConfusionMatrix(y_test['1.24'], ypred)

print(cm)

cm.plot()